In [6]:
mol_type = "6r7m"
n_mol = 2

T = 1.0
ε = 0.00025
L = 6
β = 1.0 / T

σ_r = 0.75
σ_t = 2.0

rs = 1.4
η = 0.3665
overlap_jump = 0.0
overlap_slope = 1.1
delaunay_eps = 100.0
comment = ""
comment = replace(comment, " " => "_")
bounds = 200.0

simulation_time_minutes = 2.0 * 60.0

in_folder = "../../Data/hpc_out/ma/$(n_mol)_$(mol_type)/1/"

"../../Data/hpc_out/ma/2_6r7m/1/"

In [7]:
files = filter(x -> occursin(".jld2", x), readdir("../../$(in_folder)"))

100-element Vector{String}:
 "100_rwm_ma_2_6r7m.jld2"
 "10_rwm_ma_2_6r7m.jld2"
 "11_rwm_ma_2_6r7m.jld2"
 "12_rwm_ma_2_6r7m.jld2"
 "13_rwm_ma_2_6r7m.jld2"
 "14_rwm_ma_2_6r7m.jld2"
 "15_rwm_ma_2_6r7m.jld2"
 "16_rwm_ma_2_6r7m.jld2"
 "17_rwm_ma_2_6r7m.jld2"
 "18_rwm_ma_2_6r7m.jld2"
 "19_rwm_ma_2_6r7m.jld2"
 "1_rwm_ma_2_6r7m.jld2"
 "20_rwm_ma_2_6r7m.jld2"
 ⋮
 "8_rwm_ma_2_6r7m.jld2"
 "90_rwm_ma_2_6r7m.jld2"
 "91_rwm_ma_2_6r7m.jld2"
 "92_rwm_ma_2_6r7m.jld2"
 "93_rwm_ma_2_6r7m.jld2"
 "94_rwm_ma_2_6r7m.jld2"
 "95_rwm_ma_2_6r7m.jld2"
 "96_rwm_ma_2_6r7m.jld2"
 "97_rwm_ma_2_6r7m.jld2"
 "98_rwm_ma_2_6r7m.jld2"
 "99_rwm_ma_2_6r7m.jld2"
 "9_rwm_ma_2_6r7m.jld2"

In [3]:
input_specifier = "hmc_refinement_$(n_mol)_$(mol_type)"
out_folder = "../Simulations/$(input_specifier)/"
open("../configs/$(input_specifier)_config.txt", "w") do io
    i = 1
    println(io,"ArrayTaskID input_string")
    for file in files
        input_string = escape_string("file=\"$file\";simulation_time_minutes=$simulation_time_minutes;in_folder=\"$in_folder\";out_folder=\"$out_folder\";T=$(T);rs=$rs;η=$η;L=$L;ε=$ε;σ_t=$σ_t;σ_r=$σ_r;overlap_jump=$overlap_jump;overlap_slope=$overlap_slope;bnds=$(bounds);n_mol=$n_mol;mol_type=\"$mol_type\";simulation_time_minutes=$simulation_time_minutes;delaunay_eps=$delaunay_eps;comment=\"$comment\"")
        println(io, "$i $input_string")
        i += 1
    end
end


In [4]:
total_simulations = length(readlines("../configs/$(input_specifier)_config.txt")) - 1

0

In [5]:
hours = Int(round(simulation_time_minutes / 60.0))
days = hours ÷ 24
remaining_hours = hours % 24
remaining_hours_string = remaining_hours < 10 ? "0$(remaining_hours)" : string(remaining_hours)
buffer_time_string = simulation_time_minutes < 5 ? "0$(Int(simulation_time_minutes)+2)" : "30"

open("../$(input_specifier)_hmc_script.job", "w") do io
    println(io, "#!/bin/bash")
    println(io, "#SBATCH --job-name=SolvationSimulations")
    println(io, "#SBATCH --time=0$(days)-$(remaining_hours_string):$(buffer_time_string)")
    println(io, "#SBATCH --ntasks=1")
    println(io, "#SBATCH --cpus-per-task=1")
    println(io, "#SBATCH --array=1-$(total_simulations)")
    println(io, "#SBATCH --chdir=/work/spirandelli/MorphoMolHPC/")
    println(io, "#SBATCH -o ./job_log/%a.out # STDOUT")
    println(io, "")
    println(io, "export http_proxy=http://proxy2.uni-potsdam.de:3128 #Setting proxy, due to lack of Internet on compute nodes.")
    println(io, "export https_proxy=http://proxy2.uni-potsdam.de:3128")
    println(io, "")
    println(io, "module purge")
    println(io, "module load lang/Julia/1.7.3-linux-x86_64")
    println(io, "")
    println(io, "# Specify the path to the config file")
    println(io, "config=hpc_scripts/configs/$(input_specifier)_config.txt")
    println(io, "")
    println(io, "# Extract the variables from config file")
    println(io, "config_string=\$(awk -v ArrayTaskID=\$SLURM_ARRAY_TASK_ID '\$1==ArrayTaskID {print \$2}' \$config)")
    println(io, "")
    println(io, "julia -e \"$(escape_string("include(\"julia_scripts/hmc_refinement_of_rwm_run_minimal_states_call.jl\"); hmc_refinement_call(\"\$config_string\")"))\"")
end